# Scene Occlusion

In [ ]:
import os
# Now safe to import everything else
import trimesh
import numpy as np
import open3d as o3d
from pathlib import Path
import sys
sys.path.insert(0, '../')
import drm
import copy

DATASET_DIR = Path(r"../../../datasets/V-Scan/data")   # <-- change this
FOLDER_NAME = "Industrial_1_Leica-P30_1775814741137"
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"
VOXEL_SIZE = 0.05
# Distance threshold: the max coverage distance (metres)
THRESHOLD_RESOLUTION = 0.1

%load_ext autoreload
%autoreload 2


In [ ]:
# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / FOLDER_NAME / REFERENCE_NAME
empty_scene_path = dataset_dir / FOLDER_NAME / EMPTY_SCENE_NAME
var_paths = sorted((dataset_dir/FOLDER_NAME).glob(VARIATION_GLOB))
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
emptyPcd,_ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
emptyPcd = emptyPcd.voxel_down_sample(VOXEL_SIZE)
ref_scan_pos = drm.read_transform_matrix(ref_path, apply_unity_conversion=True)[:3,3]
movedRefPcd = copy.deepcopy(refPcd).translate(-ref_scan_pos)

varPcds = []
varPosses = []
for var_path in var_paths:
    varPcd,_ = drm.txt_pcd_to_open3d(var_path, apply_unity_conversion=True)
    varPcd = varPcd.voxel_down_sample(VOXEL_SIZE)
    posTransform = drm.read_transform_matrix(var_path, apply_unity_conversion=True)[:3,3]  # just to check that the transform matrix is read correctly (it is)
    varPcds.append(varPcd)
    varPosses.append(posTransform)

In [ ]:
movedVarPcd = copy.deepcopy(varPcds[0]).translate(-varPosses[0])
newScene = drm.visualise_open3d(movedVarPcd)
newScene.add_geometry(trimesh.creation.axis(origin_size=0.05, axis_length=1.0, axis_radius=0.005))
newScene.show()

In [ ]:
newScene.add_geometry(drm.visualise_open3d(movedVarPcd.get_minimal_oriented_bounding_box(), random_color=True))
newScene.show()

In [ ]:
def build_occlusion_grid(
    reference: o3d.geometry.PointCloud,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
) -> tuple[o3d.geometry.VoxelGrid, o3d.geometry.VoxelGrid]:
    """
    Builds occupied and occluded voxel grids from a reference point cloud.

    Internally shifts the cloud so the scanner is at the origin, rotates into
    the OBB local frame for compact voxelization, then for every voxel in the
    grid that is NOT occupied, marches from the scanner toward that voxel to
    determine whether an occupied voxel blocks the line of sight (occlusion).
    Both grids are returned in world space.

    Parameters
    ----------
    reference   : point cloud that defines the geometry (e.g. var_pcd)
    scanner_pos : world-space position of the scanner that captured reference
    voxel_size  : edge length of each voxel in metres

    Returns
    -------
    occupied_grid : VoxelGrid — voxels containing at least one point
    occluded_grid : VoxelGrid — empty voxels in the shadow behind geometry
    """
    pts_shifted = np.asarray(reference.points) - scanner_pos

    shifted_pcd        = o3d.geometry.PointCloud()
    shifted_pcd.points = o3d.utility.Vector3dVector(pts_shifted)
    obb    = shifted_pcd.get_minimal_oriented_bounding_box()
    R      = np.asarray(obb.R)
    center = np.asarray(obb.center)

    pts_local = (pts_shifted - center) @ R
    min_bound = pts_local.min(axis=0)
    max_bound = pts_local.max(axis=0)
    grid_size = np.floor((max_bound - min_bound) / voxel_size).astype(int) + 1

    voxel_indices = np.floor((pts_local - min_bound) / voxel_size).astype(int)
    occupied_set  = set(map(tuple, voxel_indices))

    origin_local = (np.zeros(3) - center) @ R
    origin_voxel = (origin_local - min_bound) / voxel_size

    # Build the full set of candidate voxels (everything not occupied)
    all_indices = [
        (x, y, z)
        for x in range(grid_size[0])
        for y in range(grid_size[1])
        for z in range(grid_size[2])
    ]

    occluded_set = set()
    for voxel in all_indices:
        if voxel in occupied_set:
            continue  # occupied voxels are never occluded

        voxel_center = np.array(voxel, dtype=float) + 0.5
        ray_dir      = voxel_center - origin_voxel
        ray_length   = np.linalg.norm(ray_dir)
        if ray_length == 0:
            continue
        ray_dir_n = ray_dir / ray_length

        # March from scanner toward this voxel in steps of ~half a voxel.
        # If we hit an occupied voxel before arriving, this voxel is occluded.
        step  = 0.5                  # sub-voxel step to avoid skipping thin geometry
        t     = step
        occluded = False
        while t < ray_length - step: # stop just before the target voxel
            sample = np.floor(origin_voxel + t * ray_dir_n).astype(int)
            if np.any(sample < 0) or np.any(sample >= grid_size):
                break
            if tuple(sample) in occupied_set:
                occluded = True
                break
            t += step

        if occluded:
            occluded_set.add(voxel)

    def set_to_voxel_grid(voxel_set: set, color: list) -> o3d.geometry.VoxelGrid:
        indices       = np.array(list(voxel_set))
        centres_world = (indices + 0.5) * voxel_size + min_bound
        centres_world = centres_world @ R.T + center + scanner_pos
        pcd           = o3d.geometry.PointCloud()
        pcd.points    = o3d.utility.Vector3dVector(centres_world)
        pcd.paint_uniform_color(color)
        return o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size)

    occupied_grid = set_to_voxel_grid(occupied_set, [0.86, 0.24, 0.24])
    occluded_grid = set_to_voxel_grid(occluded_set, [0.24, 0.47, 0.86])

    return occupied_grid, occluded_grid


def get_invisible_points_grid(
    points: o3d.geometry.PointCloud,
    reference: o3d.geometry.PointCloud,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
) -> tuple[o3d.geometry.PointCloud, o3d.geometry.PointCloud]:
    """
    Classifies each point in `points` as invisible (occluded or inside geometry)
    or visible, using the occlusion grid built from `reference`.

    Parameters
    ----------
    points      : point cloud to classify (e.g. uncovered candidate points)
    reference   : point cloud that defines the geometry (e.g. var_pcd)
    scanner_pos : world-space position of the scanner that captured reference
    voxel_size  : must match the value used to build the grid

    Returns
    -------
    invisible : points inside occupied or occluded voxels
    visible   : remaining points
    """
    occupied_grid, occluded_grid = build_occlusion_grid(reference, scanner_pos, voxel_size)

    pts_world      = o3d.utility.Vector3dVector(np.asarray(points.points))
    in_occupied    = np.asarray(occupied_grid.check_if_included(pts_world))
    in_occluded    = np.asarray(occluded_grid.check_if_included(pts_world))
    invisible_mask = in_occupied | in_occluded

    invisible = points.select_by_index(np.where(invisible_mask)[0])
    visible   = points.select_by_index(np.where(~invisible_mask)[0])
    return invisible, visible


def visualise_occlusion_grid(
    occupied_grid: o3d.geometry.VoxelGrid,
    occluded_grid: o3d.geometry.VoxelGrid,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
    show_occupied: bool = True,
    show_occluded: bool = True,
    show_scanner: bool = True,
) -> list[o3d.geometry.Geometry]:
    """
    Returns a list of native o3d geometries ready for show_geometries() or
    o3d.visualization.draw_geometries().

    Parameters
    ----------
    occupied_grid : first return value of build_occlusion_grid
    occluded_grid : second return value of build_occlusion_grid
    scanner_pos   : world-space scanner position, used to place the marker sphere
    voxel_size    : used to size the scanner marker sphere
    """
    geometries = []
    if show_occupied:
        geometries.append(occupied_grid)
    if show_occluded:
        geometries.append(occluded_grid)
    if show_scanner:
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=voxel_size * 1.5)
        sphere.translate(scanner_pos)
        sphere.paint_uniform_color([1.0, 0.85, 0.0])
        sphere.compute_vertex_normals()
        geometries.append(sphere)
    return geometries

In [ ]:
sys.path.insert(0, '../')
import drm
import drm.combine
occupied_grid, occluded_grid = build_occlusion_grid(movedRefPcd, [0,0,0], voxel_size=0.2)

#invisible, visible = get_invisible_points_grid(uncovered_points, var_pcd, scanner_pos)


In [ ]:
cloud = drm.visualise_open3d(movedRefPcd)
geom = visualise_occlusion_grid(None, occluded_grid, [0,0,0], voxel_size=0.5)
cloud.add_geometry(drm.visualise_open3d(geom[1:]))
cloud.show()

In [ ]:
occupied_grid, occluded_grid = build_occlusion_grid(varPcds[0], varPosses[0], voxel_size=0.1)


In [ ]:
geom = visualise_occlusion_grid(occupied_grid, occluded_grid, varPosses[0], voxel_size=0.1)
drm.visualise_open3d(geom).show()

In [ ]:

# Optionally overlay the point cloud
cloud = drm.o3d_pointcloud_to_trimesh(varPcds[0])
scene.add_geometry(cloud, node_name="cloud")

scene.show()

## Experiments

In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path
import trimesh
import sys
sys.path.insert(0, '../')
import drm
import drm.combine
import copy

DATASET_DIR = Path(r"../../../datasets/V-Scan/data")   # <-- change this
FOLDER_NAME = "bedroom_Leica-P30_1775809921180"
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"
VOXEL_SIZE = 0.05
# Distance threshold: the max coverage distance (metres)
THRESHOLD_RESOLUTION = 0.1

%load_ext autoreload
%autoreload 2


In [ ]:
# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / FOLDER_NAME / REFERENCE_NAME
empty_scene_path = dataset_dir / FOLDER_NAME / EMPTY_SCENE_NAME
var_paths = sorted((dataset_dir/FOLDER_NAME).glob(VARIATION_GLOB))
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
emptyPcd,_ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
emptyPcd = emptyPcd.voxel_down_sample(VOXEL_SIZE)
ref_scan_pos = drm.read_transform_matrix(ref_path, apply_unity_conversion=True)[:3,3]

In [ ]:
occupied_vox, occluded_vox = drm.combine.build_occlusion_grid(refPcd, ref_scan_pos, voxel_size=VOXEL_SIZE*2)
drm.visualise_open3d(drm.combine.visualise_occlusion_grid(occupied_vox, occluded_vox, ref_scan_pos, voxel_size=VOXEL_SIZE*2)).show()

In [ ]:
import numpy as np
import open3d as o3d
from dataclasses import dataclass

# ── Config ────────────────────────────────────────────────────────────────────
ANGLE_MARGIN    = 2    # degrees – azimuth / elevation matching tolerance
# ─────────────────────────────────────────────────────────────────────────────


@dataclass
class PolarPoint:
    r:        float   # radial distance from scanner origin  [metres]
    az:       float   # azimuth  (XY-plane angle from +X)    [degrees]
    el:       float   # elevation (angle from XY-plane)      [degrees]
    xyz:      np.ndarray  # original Cartesian for reference


def cartesian_to_polar(points: np.ndarray, origin: np.ndarray) -> np.ndarray:
    """
    Convert Nx3 Cartesian points to polar (r, azimuth_deg, elevation_deg)
    relative to `origin`.

    Returns Nx3 array: columns = [r, azimuth°, elevation°]
    """
    rel  = points - origin                          # translate to scanner frame
    r    = np.linalg.norm(rel, axis=1)

    az   = np.degrees(np.arctan2(rel[:, 1], rel[:, 0]))          # [-180, 180]
    el   = np.degrees(np.arcsin(np.clip(rel[:, 2] / r, -1, 1)))  # [-90,   90]

    return np.column_stack([r, az, el])


def isolate_new_points(
    refPcd:   o3d.geometry.PointCloud,
    emptyPcd: o3d.geometry.PointCloud,
    threshold: float = VOXEL_SIZE,
) -> tuple[o3d.geometry.PointCloud, np.ndarray]:
    """
    Return points in refPcd whose nearest neighbour in emptyPcd is farther
    than `threshold`.  Uses Open3D's KD-tree for efficiency.

    Returns
    -------
    new_pcd   : PointCloud containing only the isolated points
    new_mask  : boolean array (len = len(refPcd)) – True for kept points
    """
    ref_pts   = np.asarray(refPcd.points)
    empty_kd  = o3d.geometry.KDTreeFlann(emptyPcd)

    distances = np.empty(len(ref_pts))
    for i, pt in enumerate(ref_pts):
        _, _, dist_sq = empty_kd.search_knn_vector_3d(pt, 1)
        distances[i]  = np.sqrt(dist_sq[0])

    new_mask = distances > threshold
    new_pcd  = refPcd.select_by_index(np.where(new_mask)[0])
    return new_pcd, new_mask


def check_voxel_coverage(
    new_pcd:      o3d.geometry.PointCloud,
    voxel_grid:   o3d.geometry.VoxelGrid,
    ref_scan_pos: np.ndarray,
    angle_margin: float = ANGLE_MARGIN,
) -> dict:
    """
    For each voxel in `voxel_grid`, check whether any point in `new_pcd`:
      - is angularly close (az & el within ±angle_margin degrees)
      - has a SMALLER radial distance (i.e. the point is in front of the voxel)

    A voxel is "covered" when at least one such point exists — meaning the
    voxel lies behind a detected object and is occluded from the scanner.

    Returns
    -------
    {
        "covered_voxel_grid"   : o3d.geometry.VoxelGrid,
        "uncovered_voxel_grid" : o3d.geometry.VoxelGrid,
        "covered_pct"          : float,
        "uncovered_pct"        : float,
        "n_covered"            : int,
        "n_uncovered"          : int,
        "n_total"              : int,
    }
    """
    voxels = voxel_grid.get_voxels()
    n_total = len(voxels)

    empty_result = {
        "covered_voxel_grid":   o3d.geometry.VoxelGrid(),
        "uncovered_voxel_grid": o3d.geometry.VoxelGrid(),
        "covered_pct":    0.0,
        "uncovered_pct":  100.0,
        "n_covered":   0,
        "n_uncovered": 0,
        "n_total":     0,
    }

    if n_total == 0 or len(new_pcd.points) == 0:
        return empty_result

    # ── Polar coords of all voxel centres  (Mx3) ───────────────────────────
    voxel_centres = np.array([
        voxel_grid.get_voxel_center_coordinate(v.grid_index)
        for v in voxels
    ])
    polar_vox = cartesian_to_polar(voxel_centres, ref_scan_pos)  # [r, az, el]

    # ── Polar coords of the isolated new points  (Nx3) ─────────────────────
    new_pts   = np.asarray(new_pcd.points)
    polar_pts = cartesian_to_polar(new_pts, ref_scan_pos)        # [r, az, el]

    # ── Vectorised matching: for every voxel find covering points ───────────
    # Shapes: polar_vox (M,3), polar_pts (N,3)
    # Broadcast to (M, N) for angular diff arrays.
    az_diff = np.abs(polar_vox[:, 1, None] - polar_pts[None, :, 1])
    az_diff = np.minimum(az_diff, 360.0 - az_diff)              # wrap-around
    el_diff = np.abs(polar_vox[:, 2, None] - polar_pts[None, :, 2])

    angle_ok = (az_diff <= angle_margin) & (el_diff <= angle_margin)  # (M, N)
    depth_ok = polar_pts[None, :, 0] < polar_vox[:, 0, None]          # point in front of voxel

    covered_mask = np.any(angle_ok & depth_ok, axis=1)                # (M,)

    # ── Split voxel centres into covered / uncovered ────────────────────────
    covered_centres   = voxel_centres[covered_mask]
    uncovered_centres = voxel_centres[~covered_mask]

    def centres_to_voxelgrid(centres: np.ndarray) -> o3d.geometry.VoxelGrid:
        if len(centres) == 0:
            return o3d.geometry.VoxelGrid()
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(centres)
        return o3d.geometry.VoxelGrid.create_from_point_cloud(
            pcd, voxel_size=voxel_grid.voxel_size
        )

    covered_vg   = centres_to_voxelgrid(covered_centres)
    uncovered_vg = centres_to_voxelgrid(uncovered_centres)

    n_covered    = int(covered_mask.sum())
    n_uncovered  = n_total - n_covered
    covered_pct  = round(100.0 * n_covered   / n_total, 2)
    uncovered_pct= round(100.0 * n_uncovered / n_total, 2)

    return {
        "covered_voxel_grid":   covered_vg,
        "uncovered_voxel_grid": uncovered_vg,
        "covered_pct":    covered_pct,
        "uncovered_pct":  uncovered_pct,
        "n_covered":   n_covered,
        "n_uncovered": n_uncovered,
        "n_total":     n_total,
    }


def run_pipeline(
    refPcd:       o3d.geometry.PointCloud,
    emptyPcd:     o3d.geometry.PointCloud,
    ref_scan_pos: np.ndarray,
    voxel_grid:   o3d.geometry.VoxelGrid,
    threshold:    float = VOXEL_SIZE,
    angle_margin: float = ANGLE_MARGIN,
) -> dict:
    """
    End-to-end pipeline:
      1. Isolate points in refPcd not present in emptyPcd
      2. Convert to polar coordinates from ref_scan_pos
      3. Check what fraction are occluded by a voxel behind them

    Returns all intermediate and final results.
    """
    # Step 1 – isolate new points
    new_pcd, new_mask = isolate_new_points(refPcd, emptyPcd, threshold)
    print(f"[1] Isolated {len(new_pcd.points):,} / {len(refPcd.points):,} "
          f"points  (threshold={threshold} m)")

    # Step 2 – polar coordinates
    new_pts   = np.asarray(new_pcd.points)
    polar_pts = cartesian_to_polar(new_pts, ref_scan_pos)
    print(f"[2] Converted to polar  "
          f"r∈[{polar_pts[:,0].min():.2f}, {polar_pts[:,0].max():.2f}] m  "
          f"az∈[{polar_pts[:,1].min():.1f}°, {polar_pts[:,1].max():.1f}°]  "
          f"el∈[{polar_pts[:,2].min():.1f}°, {polar_pts[:,2].max():.1f}°]")

    # Step 3 – voxel coverage
    coverage = check_voxel_coverage(new_pcd, voxel_grid, ref_scan_pos, angle_margin)
    print(f"[3] Voxel coverage  →  "
          f"{coverage['covered_pct']}% covered ({coverage['n_covered']:,} voxels)  |  "
          f"{coverage['uncovered_pct']}% uncovered ({coverage['n_uncovered']:,} voxels)  "
          f"(angle margin=±{angle_margin}°)")

    return {
        "new_pcd":               new_pcd,
        "new_mask":              new_mask,
        "polar_pts":             polar_pts,
        "covered_voxel_grid":    coverage["covered_voxel_grid"],
        "uncovered_voxel_grid":  coverage["uncovered_voxel_grid"],
        "covered_pct":           coverage["covered_pct"],
        "uncovered_pct":         coverage["uncovered_pct"],
        "n_covered":             coverage["n_covered"],
        "n_uncovered":           coverage["n_uncovered"],
        "n_total":               coverage["n_total"],
    }



results = run_pipeline(refPcd, emptyPcd, ref_scan_pos, occluded_vox)
print("\nSummary:")
print(f"  New points      : {len(results['new_pcd'].points):,}")
print(f"  Total voxels    : {results['n_total']:,}")
print(f"  Covered         : {results['covered_pct']}%  ({results['n_covered']:,})")
print(f"  Uncovered       : {results['uncovered_pct']}%  ({results['n_uncovered']:,})")

In [ ]:
drm.visualise_open3d(drm.combine.visualise_occlusion_grid(None, results['uncovered_voxel_grid'], ref_scan_pos, voxel_size=VOXEL_SIZE*2)).show()

In [ ]:
"""
evaluate_dataset.py
───────────────────
Full-dataset occlusion coverage evaluation.

For every scene folder in DATASET_DIR:
  1. Load ref + empty PCD, build occlusion voxel grid
  2. Isolate new points in ref vs empty scene
  3. Check what % of occluded voxels are covered by those points
  4. Aggregate results per scan class, dump summary CSV + console report
"""

from __future__ import annotations

import traceback
from pathlib import Path

import numpy as np
import open3d as o3d
import pandas as pd

import drm
import drm.combine

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_DIR          = Path(r"../../../datasets/V-Scan/data")
FOLDER_GLOB          = "*"
REFERENCE_NAME       = "main.txt"
EMPTY_SCENE_NAME     = "main_empty.txt"

VOXEL_SIZE           = 0.05   # metres
THRESHOLD_RESOLUTION = 0.035    # distance threshold for isolating new points
ANGLE_MARGIN         = 5    # degrees

OUTPUT_CSV           = Path(r"/home/jvermandere/projects/DRM/_output/occlusion_results/occl_results.csv")
# ─────────────────────────────────────────────────────────────────────────────


def get_scan_class(pcd_path: Path) -> str:
    """Extract class by stripping the trailing _<numbers> from the folder name."""
    folder_name = pcd_path.parent.name
    parts = folder_name.rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        return parts[0]
    return folder_name


def evaluate_scene(folder: Path) -> dict | None:
    """
    Run the full pipeline for one scene folder.
    Returns a flat result dict, or None if files are missing / an error occurs.
    """
    ref_path   = folder / REFERENCE_NAME
    empty_path = folder / EMPTY_SCENE_NAME

    if not ref_path.exists() or not empty_path.exists():
        print(f"  [SKIP] missing required files")
        return None

    try:
        refPcd, _   = drm.txt_pcd_to_open3d(ref_path,   apply_unity_conversion=True)
        refPcd      = refPcd.voxel_down_sample(VOXEL_SIZE)

        emptyPcd, _ = drm.txt_pcd_to_open3d(empty_path, apply_unity_conversion=True)
        emptyPcd    = emptyPcd.voxel_down_sample(VOXEL_SIZE)

        ref_scan_pos = drm.read_transform_matrix(
            ref_path, apply_unity_conversion=True
        )[:3, 3]

        # Build occlusion grid from the reference scan
        _, occluded_vox = drm.combine.build_occlusion_grid(
            refPcd, ref_scan_pos, voxel_size=VOXEL_SIZE * 2
        )

        # Isolate points in ref that are absent from the empty scene
        new_pcd, _ = isolate_new_points(refPcd, emptyPcd, THRESHOLD_RESOLUTION)

        # Check what fraction of occluded voxels are covered by those points
        cov = check_voxel_coverage(
            new_pcd, occluded_vox, ref_scan_pos, ANGLE_MARGIN
        )

        print(
            f"  occ_voxels={len(occluded_vox.get_voxels()):>6,}  "
            f"new_pts={len(new_pcd.points):>6,}  "
            f"covered={cov['covered_pct']:>6.2f}%  "
            f"uncovered={cov['uncovered_pct']:>6.2f}%"
        )

        return {
            "scene":         folder.name,
            "scan_class":    get_scan_class(ref_path),
            "n_ref_pts":     len(refPcd.points),
            "n_new_points":  len(new_pcd.points),
            "n_total_vox":   cov["n_total"],
            "n_covered":     cov["n_covered"],
            "n_uncovered":   cov["n_uncovered"],
            "covered_pct":   cov["covered_pct"],
            "uncovered_pct": cov["uncovered_pct"],
        }

    except Exception:
        print(f"  [ERROR]")
        traceback.print_exc()
        return None


def run_evaluation() -> pd.DataFrame:
    folders = sorted(f for f in DATASET_DIR.glob(FOLDER_GLOB) if f.is_dir())

    if not folders:
        raise FileNotFoundError(f"No folders found under {DATASET_DIR}")

    rows: list[dict] = []

    for folder in folders:
        print(f"\n{'─'*60}")
        print(f"Scene : {folder.name}")
        result = evaluate_scene(folder)
        if result:
            rows.append(result)

    return pd.DataFrame(rows)


def print_class_summary(df: pd.DataFrame) -> None:
    if df.empty:
        print("\n[!] No results to summarise.")
        return

    numeric = ["n_new_points", "n_total_vox", "n_covered", "n_uncovered",
               "covered_pct", "uncovered_pct"]

    summary = df.groupby("scan_class")[numeric].agg(["mean", "std", "count"]).round(2)

    print(f"\n{'═'*60}")
    print("CLASS SUMMARY  (mean ± std over all scenes in class)")
    print(f"{'═'*60}")

    for cls in summary.index:
        n        = int(summary.loc[cls, ("covered_pct", "count")])
        cov_mean = summary.loc[cls, ("covered_pct", "mean")]
        cov_std  = summary.loc[cls, ("covered_pct", "std")]
        unc_mean = summary.loc[cls, ("uncovered_pct", "mean")]
        unc_std  = summary.loc[cls, ("uncovered_pct", "std")]
        pts_mean = summary.loc[cls, ("n_new_points", "mean")]

        print(
            f"  {cls:<35}  n={n:>3}  "
            f"covered={cov_mean:>6.2f}%±{cov_std:>5.2f}  "
            f"uncovered={unc_mean:>6.2f}%±{unc_std:>5.2f}  "
            f"avg_new_pts={pts_mean:>7.0f}"
        )



df = run_evaluation()

if not df.empty:
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n[✓] Results saved → {OUTPUT_CSV}")

print_class_summary(df)

## Rendereing occlusion grid

In [ ]:
import os
# Now safe to import everything else
import trimesh
import numpy as np
import open3d as o3d
from pathlib import Path
import sys
sys.path.insert(0, '../')
import drm
import drm.combine
import copy

DATASET_DIR = Path(r"../../../datasets/V-Scan/data")   # <-- change this
FOLDER_NAME = "Industrial_1_Leica-P30_1775814741137" #"Office_1_Leica-P30_1775812437673" #
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"
VOXEL_SIZE = 0.05
# Distance threshold: the max coverage distance (metres)
THRESHOLD_RESOLUTION = 0.1

%load_ext autoreload
%autoreload 2

# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / FOLDER_NAME / REFERENCE_NAME
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
ref_scan_pos = drm.read_transform_matrix(ref_path, apply_unity_conversion=True)[:3,3]
movedRefPcd = copy.deepcopy(refPcd).translate(-ref_scan_pos)
occupied_grid, occluded_grid = drm.combine.build_occlusion_grid(movedRefPcd, [0,0,0], voxel_size=0.2)

In [ ]:
"""
Isometric Scene Renderer (trimesh native)
Renders a trimesh scene containing point clouds, meshes, and Open3D VoxelGrids
from an isometric viewpoint. Uses trimesh's own rendering pipeline.

Usage in Jupyter:
    from render_scene import render_scene, show

    import trimesh
    import open3d as o3d

    scene = trimesh.Scene()
    scene.add_geometry(my_pointcloud, node_name="scan")

    # Render with an Open3D VoxelGrid overlaid
    show(scene, voxel_grids=[my_voxelgrid], size=1024,
         clip_ceiling_pct=90, clip_x_pct=15, clip_z_pct=15)

    # Or get the PIL image back
    img = render_scene(scene, voxel_grids=[my_voxelgrid], size=1024)
    img.save("render.png")

Also accepts raw trimesh geometry (PointCloud, Trimesh) — it will be
wrapped in a Scene automatically.
"""

import io
import math
from typing import Optional, Union, List

import numpy as np
import trimesh
from PIL import Image


def _clip_pointcloud(
    cloud: trimesh.PointCloud,
    z_max: float,
    x_min: float,
    y_min: float,
) -> trimesh.PointCloud:
    """Clip a point cloud. Z is up, camera looks toward +X/+Y."""
    pts = np.array(cloud.vertices)
    mask = (pts[:, 2] <= z_max) & (pts[:, 0] >= x_min) & (pts[:, 1] >= y_min)

    colors = None
    if cloud.colors is not None and len(cloud.colors) == len(pts):
        colors = np.array(cloud.colors)[mask]

    return trimesh.PointCloud(vertices=pts[mask], colors=colors)


def _clip_mesh(
    mesh: trimesh.Trimesh,
    z_max: float,
    x_min: float,
    y_min: float,
) -> trimesh.Trimesh:
    """
    Clip a mesh by removing faces where all three vertices fall outside
    any clipping plane. Z is up, camera looks toward +X/+Y.
    """
    verts = np.array(mesh.vertices)
    outside = (verts[:, 2] >= z_max) | (verts[:, 0] < x_min) | (verts[:, 1] < y_min)

    faces = np.array(mesh.faces)
    face_mask = ~outside[faces].all(axis=1)

    clipped = mesh.submesh([np.where(face_mask)[0]], append=True)
    return clipped


def _compute_clip_thresholds(
    bbox_min: np.ndarray,
    bbox_max: np.ndarray,
    clip_ceiling_pct: float,
    clip_x_pct: float,
    clip_z_pct: float,
    abs_ceiling: Optional[float],
    abs_x: Optional[float],
    abs_z: Optional[float],
) -> tuple:
    """Compute absolute clip thresholds. Z is up, camera looks toward +X/+Y."""
    bbox_range = bbox_max - bbox_min

    z_max = abs_ceiling if abs_ceiling is not None else bbox_min[2] + bbox_range[2] * (clip_ceiling_pct / 100.0)
    x_min = abs_x if abs_x is not None else bbox_min[0] + bbox_range[0] * (clip_x_pct / 100.0)
    y_min = abs_z if abs_z is not None else bbox_min[1] + bbox_range[1] * (clip_z_pct / 100.0)

    return z_max, x_min, y_min


def _ensure_scene(geom: Union[trimesh.Scene, trimesh.PointCloud, trimesh.Trimesh]) -> trimesh.Scene:
    """Wrap raw geometry in a Scene if needed."""
    if isinstance(geom, trimesh.Scene):
        return geom
    scene = trimesh.Scene()
    scene.add_geometry(geom)
    return scene


def _voxelgrid_to_trimesh(voxel_grid, alpha: int = 180) -> trimesh.Trimesh:
    """
    Convert an Open3D VoxelGrid to a trimesh Trimesh (collection of cubes).
    Each voxel becomes a box at its grid position with its color.
    """
    voxels = voxel_grid.get_voxels()
    if len(voxels) == 0:
        return trimesh.Trimesh()

    voxel_size = voxel_grid.voxel_size
    origin = np.array(voxel_grid.origin)

    all_meshes = []
    for voxel in voxels:
        idx = np.array(voxel.grid_index, dtype=np.float64)
        center = origin + (idx + 0.5) * voxel_size

        box = trimesh.creation.box(
            extents=[voxel_size, voxel_size, voxel_size],
            transform=trimesh.transformations.translation_matrix(center),
        )

        # Get voxel color if available
        if hasattr(voxel, 'color') and voxel.color is not None:
            c = np.array(voxel.color)
            if c.max() <= 1.0:
                c = (c * 255).astype(np.uint8)
            rgba = [int(c[0]), int(c[1]), int(c[2]), alpha]
        else:
            rgba = [128, 128, 128, alpha]

        box.visual.vertex_colors = np.tile(rgba, (len(box.vertices), 1)).astype(np.uint8)
        all_meshes.append(box)

    if not all_meshes:
        return trimesh.Trimesh()

    return trimesh.util.concatenate(all_meshes)


def _clip_voxelgrid_mesh(
    mesh: trimesh.Trimesh,
    z_max: float,
    x_min: float,
    y_min: float,
) -> trimesh.Trimesh:
    """Clip a voxelgrid mesh the same way as any other mesh."""
    return _clip_mesh(mesh, z_max, x_min, y_min)


def _voxelgrid_to_points_and_colors(voxel_grid) -> tuple:
    """
    Extract voxel centers and colors from an Open3D VoxelGrid.
    Returns (centers [N,3], colors [N,4] uint8).
    """
    voxels = voxel_grid.get_voxels()
    if len(voxels) == 0:
        return np.empty((0, 3)), np.empty((0, 4), dtype=np.uint8)

    voxel_size = voxel_grid.voxel_size
    origin = np.array(voxel_grid.origin)

    centers = []
    colors = []
    for voxel in voxels:
        idx = np.array(voxel.grid_index, dtype=np.float64)
        center = origin + (idx + 0.5) * voxel_size
        centers.append(center)

        if hasattr(voxel, 'color') and voxel.color is not None:
            c = np.array(voxel.color)
            if c.max() <= 1.0:
                c = (c * 255).astype(np.uint8)
            colors.append([int(c[0]), int(c[1]), int(c[2]), 180])
        else:
            colors.append([128, 128, 128, 180])

    return np.array(centers), np.array(colors, dtype=np.uint8)


def _build_iso_camera_transform(
    center: np.ndarray,
    distance: float,
    up_axis: str = "z",
    quadrant: int = 1,
) -> np.ndarray:
    """
    Build a 4x4 camera transform matrix for isometric view.

    Args:
        center:   Scene center to look at
        distance: Camera pull-back distance
        up_axis:  Which world axis points up — 'x', 'y', or 'z'
        quadrant: Which horizontal quadrant the camera sits in (1-4).
                  Think of it as a compass when looking down the up axis:
                    1 = +A / +B  (front-right)
                    2 = -A / +B  (front-left)
                    3 = -A / -B  (back-left)
                    4 = +A / -B  (back-right)
                  where A and B are the two axes perpendicular to up_axis:
                    up=z → A=x, B=y
                    up=y → A=x, B=z
                    up=x → A=y, B=z
    """
    up_axis = up_axis.lower()

    axes = {
        "z": (np.array([1., 0., 0.]), np.array([0., 1., 0.]), np.array([0., 0., 1.])),
        "y": (np.array([1., 0., 0.]), np.array([0., 0., 1.]), np.array([0., 1., 0.])),
        "x": (np.array([0., 1., 0.]), np.array([0., 0., 1.]), np.array([1., 0., 0.])),
    }
    if up_axis not in axes:
        raise ValueError(f"up_axis must be 'x', 'y', or 'z', got '{up_axis}'")

    a, b, up = axes[up_axis]

    signs = {1: (+1, +1), 2: (-1, +1), 3: (-1, -1), 4: (+1, -1)}
    if quadrant not in signs:
        raise ValueError(f"quadrant must be 1-4, got {quadrant}")

    sa, sb = signs[quadrant]

    # Camera positioned in the chosen horizontal quadrant and above the scene
    cam_dir = sa * a + sb * b + up
    cam_dir = cam_dir / np.linalg.norm(cam_dir)

    eye = center + cam_dir * distance

    forward = center - eye
    forward = forward / np.linalg.norm(forward)

    right = np.cross(forward, up)
    right = right / np.linalg.norm(right)
    true_up = np.cross(right, forward)

    transform = np.eye(4)
    transform[:3, 0] = right
    transform[:3, 1] = true_up
    transform[:3, 2] = -forward
    transform[:3, 3] = eye

    return transform


def render_scene(
    scene: Union[trimesh.Scene, trimesh.PointCloud, trimesh.Trimesh],
    size: int = 1024,
    point_size: float = 2.0,
    bg_color: tuple = (255, 255, 255, 0),
    margin: float = 1.1,
    # Clipping
    clip_ceiling_pct: float = 100.0,
    clip_x_pct: float = 0.0,
    clip_z_pct: float = 0.0,
    abs_ceiling: Optional[float] = None,
    abs_x: Optional[float] = None,
    abs_z: Optional[float] = None,
    # Open3D VoxelGrids
    voxel_grids: Optional[List] = None,
    voxel_alpha: int = 180,
    up_axis: str = "z",
    quadrant: int = 1,
    # Lighting
    light_direction: Optional[np.ndarray] = None,
    light_intensity: tuple = (1.0, 1.0, 1.0),
    light_strength: float = 5.0,
) -> Image.Image:
    """
    Render a trimesh Scene/geometry isometrically.

    Uses trimesh's built-in rendering (pyglet/OpenGL) with scene.save_image().
    Works in Jupyter and headless environments.

    Args:
        scene:            trimesh.Scene, PointCloud, or Trimesh
        size:             Square output image size in pixels
        point_size:       Point size for point clouds
        bg_color:         Background RGBA as (0-255, 0-255, 0-255, 0-255)
        margin:           Margin factor around the scene (1.0 = tight fit)
        clip_ceiling_pct: Remove points/faces above this % of Z range (100 = no clip)
        clip_x_pct:       Remove points/faces below this % of X range (0 = no clip)
        clip_z_pct:       Remove points/faces below this % of Y range (0 = no clip)
        abs_ceiling:      Absolute Z max cutoff (overrides clip_ceiling_pct)
        abs_x:            Absolute X min cutoff (overrides clip_x_pct)
        abs_z:            Absolute Y min cutoff (overrides clip_z_pct)
        voxel_grids:      List of open3d.geometry.VoxelGrid objects to render
        voxel_alpha:      Alpha (opacity) for voxel cubes, 0-255 (default: 180)

    Returns:
        PIL.Image.Image in RGBA mode
    """
    scene = _ensure_scene(scene)

    # Convert VoxelGrids to trimesh meshes and add to scene
    if voxel_grids:
        for i, vg in enumerate(voxel_grids):
            vg_mesh = _voxelgrid_to_trimesh(vg, alpha=voxel_alpha)
            if len(vg_mesh.vertices) > 0:
                scene.add_geometry(vg_mesh, node_name=f"_voxelgrid_{i}")

    # Collect all vertices for global bounding box
    all_verts = []
    for geom in scene.geometry.values():
        if hasattr(geom, "vertices") and len(geom.vertices) > 0:
            all_verts.append(np.array(geom.vertices))

    if not all_verts:
        return Image.new("RGBA", (size, size), bg_color)

    combined = np.vstack(all_verts)
    bbox_min = combined.min(axis=0)
    bbox_max = combined.max(axis=0)

    # Clipping
    needs_clip = clip_ceiling_pct < 100 or clip_x_pct > 0 or clip_z_pct > 0 or \
                 abs_ceiling is not None or abs_x is not None or abs_z is not None

    if needs_clip:
        z_max, x_min, y_min = _compute_clip_thresholds(
            bbox_min, bbox_max,
            clip_ceiling_pct, clip_x_pct, clip_z_pct,
            abs_ceiling, abs_x, abs_z,
        )

        clipped_scene = trimesh.Scene()
        for name, geom in scene.geometry.items():
            if isinstance(geom, trimesh.PointCloud):
                geom = _clip_pointcloud(geom, z_max, x_min, y_min)
                if len(geom.vertices) > 0:
                    clipped_scene.add_geometry(geom, node_name=name)
            elif isinstance(geom, trimesh.Trimesh):
                geom = _clip_mesh(geom, z_max, x_min, y_min)
                if len(geom.vertices) > 0 and len(geom.faces) > 0:
                    clipped_scene.add_geometry(geom, node_name=name)
            else:
                clipped_scene.add_geometry(geom, node_name=name)
        scene = clipped_scene

    # Recompute bounds from clipped scene
    all_verts = []
    for geom in scene.geometry.values():
        # Define source direction (e.g., current mesh "up" axis) and target direction
        v_start = [0, 0, 1]
        v_end = [0, 1, 0]

        # Compute the rotation matrix aligning v_start to v_end
        rotation_matrix = trimesh.geometry.align_vectors(v_start, v_end)

        # Apply
        geom.apply_transform(rotation_matrix)
        if hasattr(geom, "vertices") and len(geom.vertices) > 0:
            all_verts.append(np.array(geom.vertices))

    if not all_verts:
        return Image.new("RGBA", (size, size), bg_color)

    combined = np.vstack(all_verts)
    center = combined.mean(axis=0)
    extent = np.linalg.norm(combined.max(axis=0) - combined.min(axis=0))

    # Set up isometric camera
    distance = extent * 2.0
    cam_transform = _build_iso_camera_transform(center, distance, up_axis=up_axis, quadrant=quadrant)

    # Orthographic camera: fov=0 signals orthographic in trimesh,
    # but trimesh uses a specific Camera class
    ortho_half = (extent * margin) / 2.0
    camera = trimesh.scene.Camera(
        resolution=(size, size),
        fov=(10, 10),  # orthographic
    )

    # For orthographic, trimesh uses the `camera.K` matrix.
    # We set the scale via the scene's camera transform.
    scene.camera = camera
    scene.camera_transform = cam_transform

    # Add directional light
    if light_direction is not None:
        ld = np.array(light_direction, dtype=np.float64)
        ld = ld / np.linalg.norm(ld)
    else:
        # Default: come from above and slightly in front of quadrant 1
        ld = np.array([1.0, 1.0, 2.0])
        ld = ld / np.linalg.norm(ld)
 
    light_color = np.array([int(c * 255) for c in light_intensity], dtype=np.uint8)
 
    light = trimesh.scene.lighting.DirectionalLight(
        color=light_color,
        intensity=light_strength,
    )
 
    # DirectionalLight shines along -Z in its own frame,
    # so we build a transform that rotates -Z to point along -ld (toward scene)
    # We need to orient the light so its -Z aligns with the light travel direction
    light_forward = -ld  # light travels toward the scene
    light_up = np.array([0., 0., 1.]) if abs(light_forward[2]) < 0.99 else np.array([0., 1., 0.])
    light_right = np.cross(light_up, -light_forward)
    light_right = light_right / np.linalg.norm(light_right)
    light_up = np.cross(-light_forward, light_right)
 
    light_transform = np.eye(4)
    light_transform[:3, 0] = light_right
    light_transform[:3, 1] = light_up
    light_transform[:3, 2] = -light_forward
    light_transform[:3, 3] = center + ld * distance  # position far away in source direction
 
    scene.graph.update(frame_from="world", frame_to="directional_light", matrix=light_transform)
    scene.lights = [light]
 
    return scene


def _render_matplotlib(
    scene: trimesh.Scene,
    size: int,
    bg_color: tuple,
    cam_transform: np.ndarray,
    center: np.ndarray,
    extent: float,
    margin: float,
) -> Image.Image:
    """
    Fallback renderer using matplotlib's 3D plotting.
    Renders point clouds, meshes, and voxel cubes (from converted VoxelGrids)
    all in the same 3D axes so matplotlib handles z-buffering between them.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection

    dpi = 100
    fig = plt.figure(figsize=(size / dpi, size / dpi), dpi=dpi)
    ax = fig.add_subplot(111, projection="3d")

    # Set background
    bg = [c / 255.0 for c in bg_color[:3]]
    bg_alpha = bg_color[3] / 255.0 if len(bg_color) > 3 else 1.0
    fig.patch.set_facecolor(bg + [bg_alpha])
    ax.set_facecolor(bg + [bg_alpha])

    for name, geom in scene.geometry.items():
        if isinstance(geom, trimesh.PointCloud):
            pts = np.array(geom.vertices)
            if len(pts) == 0:
                continue
            colors = None
            if geom.colors is not None and len(geom.colors) > 0:
                c = np.array(geom.colors, dtype=np.float64)
                if c.max() > 1.0:
                    c = c / 255.0
                colors = c[:, :3]
            ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=colors, s=1, depthshade=True)

        elif isinstance(geom, trimesh.Trimesh):
            verts = np.array(geom.vertices)
            faces = np.array(geom.faces)
            if len(verts) == 0 or len(faces) == 0:
                continue

            # Get face colors
            face_colors = None
            if hasattr(geom.visual, 'vertex_colors') and geom.visual.vertex_colors is not None:
                vc = np.array(geom.visual.vertex_colors, dtype=np.float64)
                if vc.max() > 1.0:
                    vc = vc / 255.0
                # Average vertex colors per face, include alpha
                fc = vc[faces].mean(axis=1)
                if fc.shape[1] >= 4:
                    face_colors = fc[:, :4]
                else:
                    face_colors = fc[:, :3]

            polys = verts[faces]

            alpha_val = 0.8
            if face_colors is not None:
                if face_colors.shape[1] == 4:
                    alpha_val = None  # per-face alpha in the colors
                collection = Poly3DCollection(polys)
                collection.set_facecolors(face_colors)
            else:
                collection = Poly3DCollection(polys, alpha=0.8)
                collection.set_facecolor([0.7, 0.7, 0.7])
            collection.set_edgecolor("none")
            ax.add_collection3d(collection)

    # Isometric view angles
    ax.view_init(elev=35.264, azim=-45)

    # Equal aspect ratio — Z is up
    half = extent * margin / 2
    ax.set_xlim(center[0] - half, center[0] + half)
    ax.set_ylim(center[1] - half, center[1] + half)
    ax.set_zlim(center[2] - half, center[2] + half)

    ax.set_axis_off()
    plt.tight_layout(pad=0)

    # Render to image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0, transparent=(bg_alpha < 1.0))
    plt.close(fig)
    buf.seek(0)

    return Image.open(buf).convert("RGBA")

def pointcloud_to_mesh(
    cloud: trimesh.PointCloud,
    point_size: float,
) -> trimesh.Trimesh:
    """
    Convert a PointCloud to a mesh of small boxes, one per point.
    This makes point size actual geometry so it renders at the correct
    size in any viewer, including trimesh's interactive pyglet window.
 
    Args:
        cloud:      Input trimesh PointCloud
        point_size: World-space size of each box (e.g. 0.01 for 1cm points)
 
    Returns:
        trimesh.Trimesh with one box per point, colored to match
    """
    pts = np.array(cloud.vertices)
    if len(pts) == 0:
        return trimesh.Trimesh()
 
    colors = None
    if cloud.colors is not None and len(cloud.colors) == len(pts):
        colors = np.array(cloud.colors, dtype=np.uint8)
 
    # Create a single template box centered at origin
    template = trimesh.creation.uv_sphere(radius=point_size / 2, count=[4, 4])
    n_verts = len(template.vertices)
    n_faces = len(template.faces)
 
    all_verts = np.empty((len(pts) * n_verts, 3), dtype=np.float64)
    all_faces = np.empty((len(pts) * n_faces, 3), dtype=np.int64)
    all_colors = np.empty((len(pts) * n_verts, 4), dtype=np.uint8)
 
    for i, pt in enumerate(pts):
        all_verts[i * n_verts:(i + 1) * n_verts] = template.vertices + pt
        all_faces[i * n_faces:(i + 1) * n_faces] = template.faces + i * n_verts
 
        if colors is not None:
            c = colors[i]
            rgba = c if len(c) == 4 else np.append(c, 255)
        else:
            rgba = np.array([128, 128, 128, 255], dtype=np.uint8)
 
        all_colors[i * n_verts:(i + 1) * n_verts] = rgba
 
    mesh = trimesh.Trimesh(vertices=all_verts, faces=all_faces, process=False)
    mesh.visual.vertex_colors = all_colors
    return mesh
    """Render a prepared trimesh scene to a PIL image via matplotlib."""
    all_verts = [np.array(g.vertices) for g in scene.geometry.values()
                 if hasattr(g, "vertices") and len(g.vertices) > 0]
    if not all_verts:
        return Image.new("RGBA", (size, size), bg_color)
    combined = np.vstack(all_verts)
    center = combined.mean(axis=0)
    extent = np.linalg.norm(combined.max(axis=0) - combined.min(axis=0))
    return _render_matplotlib(scene, size, bg_color, scene.camera_transform, center, extent, margin)
 

def show(
    scene: Union[trimesh.Scene, trimesh.PointCloud, trimesh.Trimesh],
    size: int = 1024,
    voxel_grids: Optional[List] = None,
    **kwargs,
):
    """
    Render and display inline in Jupyter notebook.
    Accepts all the same kwargs as render_scene().
    """
    from IPython.display import display
    img = render_scene(scene, size=size, voxel_grids=voxel_grids, **kwargs)
    display(img)
    return img

In [ ]:
cloud = drm.visualise_open3d(movedRefPcd.voxel_down_sample(0.2))
cloud_mesh = pointcloud_to_mesh(cloud.dump()[0], point_size=0.15)
geom = drm.combine.visualise_occlusion_grid(occupied_grid, occluded_grid, [0,0,0], voxel_size=0.2)
geom[2].paint_uniform_color((0.5, 0.5, 1))
cloud_scene = trimesh.scene.Scene(cloud_mesh)
cloud_scene.add_geometry(drm.visualise_open3d(geom[1:]))

# With wall/ceiling clipping
newScene = render_scene(
    cloud_scene,
    size=1024,
    point_size=6,
    bg_color=(0, 0, 0, 0),        # transparent
    clip_ceiling_pct=90,            # remove top 10% of Y
    clip_x_pct=10,                  # remove lowest 15% of X
    clip_z_pct=10,                  # remove lowest 15% of Z
    up_axis="y",
    quadrant=1,
    light_direction=[0,-1,0]
)
newScene.show(height=1000, flags={'fullscreen': False})

In [ ]:

# With wall/ceiling clipping
img = render_scene(
    cloud,
    size=1024,
    point_size=3.0,
    bg_color=(0, 0, 0, 0),        # transparent
    clip_ceiling_pct=90,            # remove top 10% of Y
    clip_x_pct=15,                  # remove lowest 15% of X
    clip_z_pct=15,                  # remove lowest 15% of Z
)
img.save("clipped.png")

In [ ]:
show(drm.visualise_open3d(movedRefPcd.voxel_down_sample(0.2)), voxel_grids=[occluded_grid], size=512,
     clip_ceiling_pct=90, clip_x_pct=15, clip_z_pct=15)